# Combined Feature Engineering — NYC Yellow Taxi

This notebook combines feature engineering from all four team members, using the code from each person's notebook.

| Person   | Assignment                              | Source notebook   |
|----------|----------------------------------------|-------------------|
| **Moses**   | Temporal/Time Features                  | Moses_FE.ipynb    |
| **Tarun**   | Location/Spatial Features               | Tarun_FE.ipynb    |
| **Abhishek**| Trip Characteristics & Derived Metrics  | Abhishek_FE.ipynb |
| **Morgan**  | Categorical Encodings & Interaction Features | Morgan_FE.ipynb |

Features are trimmed for **stakeholder focus** (trip duration + congestion fee prediction for drivers/fleet operators) and **smaller file size**: .

**Input:** `data/processed/processed_taxi_cleaned.parquet`  
**Output:** `data/processed/taxi_engineered.parquet`

## 1. Setup and Data Loading

Load the cleaned trip data and the TLC taxi zone lookup (used in Tarun's location features).

In [24]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

# Load cleaned trip data and zone lookup
df = pd.read_parquet("../../data/processed/processed_taxi_cleaned.parquet")
zones = pd.read_csv("../../data/raw/taxi_zone_lookup.csv")

print(f"Trips: {df.shape[0]:,} x {df.shape[1]} columns")
print(f"Zone lookup: {zones.shape[0]} zones")
df.head(3)

Trips: 2,451,103 x 22 columns
Zone lookup: 265 zones


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_duration_min,has_congestion_fee
0,2,2025-01-05 00:00:02,2025-01-05 00:17:47,1.00,8.66,1.00,N,138,43,1,35.20,6.00,0.50,12.85,6.94,1.00,64.24,0.00,1.75,0.00,17.75,0
1,2,2025-01-05 00:24:51,2025-01-05 00:50:27,1.00,9.73,1.00,N,138,61,1,40.80,6.00,0.50,10.01,0.00,1.00,60.06,0.00,1.75,0.00,25.60,0
2,2,2025-01-05 00:54:12,2025-01-05 01:17:58,1.00,10.33,1.00,N,138,62,1,41.50,6.00,0.50,10.00,0.00,1.00,60.75,0.00,1.75,0.00,23.77,0


## 2. Temporal/Time Features

From **Moses_FE.ipynb**. Basic time components (hour, day of week, month, day of month), time-based flags (weekend, peak hour, rush hour), time-of-day category, and cyclical encoding (sin/cos) for hour and day of week. Trip duration is already in the cleaned data as `trip_duration_min`.

In [25]:
# --- Moses_FE: Basic time components ---
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])
df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
df["pickup_day_of_week"] = df["tpep_pickup_datetime"].dt.dayofweek  # 0=Monday, 6=Sunday
df["pickup_day_name"] = df["tpep_pickup_datetime"].dt.day_name()
df["pickup_month"] = df["tpep_pickup_datetime"].dt.month
df["pickup_day_of_month"] = df["tpep_pickup_datetime"].dt.day

# --- Time flags ---
df["is_weekend"] = (df["pickup_day_of_week"] >= 5).astype(int)
df["is_peak_hour"] = ((df["pickup_hour"] >= 17) & (df["pickup_hour"] <= 19)).astype(int)
df["is_morning_rush"] = ((df["pickup_hour"] >= 7) & (df["pickup_hour"] <= 9)).astype(int)
df["is_evening_rush"] = ((df["pickup_hour"] >= 17) & (df["pickup_hour"] <= 19)).astype(int)
df["is_rush_hour"] = (df["is_morning_rush"] | df["is_evening_rush"]).astype(int)

# --- Time-of-day category ---
def get_time_of_day(hour):
    if 5 <= hour < 12:
        return "morning"
    elif 12 <= hour < 17:
        return "afternoon"
    elif 17 <= hour < 21:
        return "evening"
    return "night"
df["time_of_day"] = df["pickup_hour"].apply(get_time_of_day)

# --- Cyclical encoding ---
df["hour_sin"] = np.sin(2 * np.pi * df["pickup_hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["pickup_hour"] / 24)
df["day_of_week_sin"] = np.sin(2 * np.pi * df["pickup_day_of_week"] / 7)
df["day_of_week_cos"] = np.cos(2 * np.pi * df["pickup_day_of_week"] / 7)

print("Moses — Temporal features added.")
df[["pickup_hour", "pickup_day_of_week", "is_weekend", "is_rush_hour", "time_of_day"]].head()

Moses — Temporal features added.


,pickup_hour,pickup_day_of_week,is_weekend,is_rush_hour,time_of_day
0,0,6,1,0,night
1,0,6,1,0,night
2,0,6,1,0,night
3,0,6,1,0,night
4,0,6,1,0,night


## 3. Location/Spatial Features

From **Tarun_FE.ipynb**. Borough mapping from the taxi zone lookup, same-borough and borough-route labels, airport pickup/dropoff flags (JFK, LaGuardia, EWR), and zone/route frequency encoding.

In [26]:
# --- Tarun_FE: Borough mapping ---
borough_map = zones.set_index("LocationID")["Borough"].to_dict()
df["pickup_borough"] = df["PULocationID"].map(borough_map)
df["dropoff_borough"] = df["DOLocationID"].map(borough_map)
df["pickup_borough"] = df["pickup_borough"].fillna("Unknown")
df["dropoff_borough"] = df["dropoff_borough"].fillna("Unknown")

# --- Cross-borough and route ---
df["is_same_borough"] = (df["pickup_borough"] == df["dropoff_borough"]).astype(int)
df["borough_route"] = df["pickup_borough"] + " -> " + df["dropoff_borough"]

# --- Airport flags (JFK=132, LaGuardia=138, EWR=1) ---
AIRPORT_ZONES = {"JFK": 132, "LaGuardia": 138, "EWR": 1}
ALL_AIRPORT_IDS = list(AIRPORT_ZONES.values())
df["is_airport_pickup"] = df["PULocationID"].isin(ALL_AIRPORT_IDS).astype(int)
df["is_jfk_pickup"] = (df["PULocationID"] == AIRPORT_ZONES["JFK"]).astype(int)
df["is_lga_pickup"] = (df["PULocationID"] == AIRPORT_ZONES["LaGuardia"]).astype(int)
df["is_ewr_pickup"] = (df["PULocationID"] == AIRPORT_ZONES["EWR"]).astype(int)
df["is_airport_dropoff"] = df["DOLocationID"].isin(ALL_AIRPORT_IDS).astype(int)
df["is_jfk_dropoff"] = (df["DOLocationID"] == AIRPORT_ZONES["JFK"]).astype(int)
df["is_lga_dropoff"] = (df["DOLocationID"] == AIRPORT_ZONES["LaGuardia"]).astype(int)
df["is_ewr_dropoff"] = (df["DOLocationID"] == AIRPORT_ZONES["EWR"]).astype(int)
df["is_airport_trip"] = (df["is_airport_pickup"] | df["is_airport_dropoff"]).astype(int)

# --- Zone and route frequency ---
pu_freq = df["PULocationID"].value_counts(normalize=True)
do_freq = df["DOLocationID"].value_counts(normalize=True)
df["pickup_zone_freq"] = df["PULocationID"].map(pu_freq)
df["dropoff_zone_freq"] = df["DOLocationID"].map(do_freq)
df["route_id"] = df["PULocationID"].astype(str) + "_" + df["DOLocationID"].astype(str)
route_freq = df["route_id"].value_counts(normalize=True)
df["route_freq"] = df["route_id"].map(route_freq)

print("Tarun — Location/Spatial features added.")
print(f"  Same-borough: {df['is_same_borough'].mean()*100:.1f}%  |  Airport trips: {df['is_airport_trip'].mean()*100:.1f}%")

Tarun — Location/Spatial features added.
  Same-borough: 88.4%  |  Airport trips: 9.4%


## 4. Trip Characteristics & Derived Metrics

From **Abhishek_FE.ipynb**. Speed (avg_speed_mph), fare efficiency (fare_per_mile, total_per_mile), passenger features (is_single_passenger, passenger_count_binned), cost ratios (tip_percentage, tip_to_total_ratio), distance category (short/medium/long), and outlier flags (is_extreme_distance, is_extreme_fare).

In [27]:
# --- Abhishek_FE: Speed ---
df["avg_speed_mph"] = df["trip_distance"] / df["trip_duration_min"] * 60
df["avg_speed_mph"] = df["avg_speed_mph"].clip(upper=60)
df["avg_speed_mph"] = df["avg_speed_mph"].replace([np.inf, -np.inf], 0).fillna(0)

# --- Fare efficiency ---
df["fare_per_mile"] = np.where(df["trip_distance"] > 0, df["fare_amount"] / df["trip_distance"], 0)
df["fare_per_mile"] = df["fare_per_mile"].clip(0, 100)
df["total_per_mile"] = np.where(df["trip_distance"] > 0, df["total_amount"] / df["trip_distance"], 0)
df["total_per_mile"] = df["total_per_mile"].clip(0, 150)

# --- Passenger features ---
df["passenger_count"] = df["passenger_count"].fillna(df["passenger_count"].mode()[0])
df["is_single_passenger"] = (df["passenger_count"] == 1).astype(int)
def bin_passengers(count):
    if count == 1: return "solo"
    elif count == 2: return "pair"
    return "group"
df["passenger_count_binned"] = df["passenger_count"].apply(bin_passengers)

# --- Cost ratios ---
df["tip_percentage"] = np.where(df["fare_amount"] > 0, (df["tip_amount"] / df["fare_amount"]) * 100, 0)
df["tip_percentage"] = df["tip_percentage"].clip(0, 100)
df["tip_to_total_ratio"] = np.where(df["total_amount"] > 0, (df["tip_amount"] / df["total_amount"]) * 100, 0)
df["tip_to_total_ratio"] = df["tip_to_total_ratio"].clip(0, 100)

# --- Distance category (short 0-2, medium 2-5, long 5+ miles) ---
df["distance_category"] = pd.cut(df["trip_distance"], bins=[0, 2, 5, float("inf")], labels=["short", "medium", "long"])

# --- Outlier flags (95th percentile) ---
dist_95 = df["trip_distance"].quantile(0.95)
fare_95 = df["fare_amount"].quantile(0.95)
df["is_extreme_distance"] = (df["trip_distance"] > dist_95).astype(int)
df["is_extreme_fare"] = (df["fare_amount"] > fare_95).astype(int)

print("Abhishek — Trip characteristics added.")
print(f"  avg_speed_mph mean: {df['avg_speed_mph'].mean():.2f}  |  is_single_passenger: {df['is_single_passenger'].mean()*100:.1f}%")

Abhishek — Trip characteristics added.
  avg_speed_mph mean: 11.20  |  is_single_passenger: 80.9%


## 5. Categorical Encodings & Interaction Features

From **Morgan_FE.ipynb**. We keep interpretable, stakeholder-relevant features without one-hot expansion: **store_and_fwd_encoded**; **payment_name** and **ratecode_name** (labels for credit/cash, standard/JFK/etc.); **time_slot** and **hour_x_dayofweek** (time interactions); **cbd_fee_ratio**, **total_surcharges**, **surcharges_ratio**, **base_fare_ratio** (fare structure for drivers). One-hot dummies are omitted to keep file size down.

In [28]:
# --- Morgan_FE: Store-and-forward + interpretable labels (no one-hot dummies to save size) ---
df["store_and_fwd_flag"] = df["store_and_fwd_flag"].fillna("N")
df["store_and_fwd_encoded"] = (df["store_and_fwd_flag"] == "Y").astype(int)
df["RatecodeID"] = df["RatecodeID"].fillna(1.0)
payment_map = {1: "credit", 2: "cash", 3: "no_charge", 4: "dispute", 5: "unknown", 6: "voided"}
df["payment_name"] = df["payment_type"].map(payment_map).fillna("other")
ratecode_map = {1: "standard", 2: "jfk", 3: "newark", 4: "nassau", 5: "negotiated", 6: "group"}
df["ratecode_name"] = df["RatecodeID"].map(ratecode_map).fillna("other")

print("Morgan — Categorical: store_and_fwd_encoded, payment_name, ratecode_name (interpretable for stakeholders).")

Morgan — Categorical: store_and_fwd_encoded, payment_name, ratecode_name (interpretable for stakeholders).


In [29]:
# --- Morgan_FE (trimmed): Key interactions + stakeholder-relevant ratios only ---
df["hour_x_dayofweek"] = df["pickup_hour"] * df["pickup_day_of_week"]
df["time_slot"] = df["pickup_day_of_week"] * 24 + df["pickup_hour"]

def pct_of_total(num, denom, low=0, high=100):
    out = np.where(denom > 0, (num / denom) * 100, 0)
    return np.clip(out, low, high)
# CBD fee ratio is directly relevant for congestion-pricing prediction (stakeholder outcome)
df["cbd_fee_ratio"] = pct_of_total(df["cbd_congestion_fee"].fillna(0), df["total_amount"], -100, 100)
df["total_surcharges"] = (
    df["congestion_surcharge"].fillna(0) + df["Airport_fee"].fillna(0) + df["mta_tax"].fillna(0)
    + df["cbd_congestion_fee"].fillna(0) + df["tolls_amount"].fillna(0) + df["extra"].fillna(0)
    + df["improvement_surcharge"].fillna(0)
)
df["surcharges_ratio"] = pct_of_total(df["total_surcharges"], df["total_amount"], 0, 100)
df["base_fare_ratio"] = pct_of_total(df["fare_amount"], df["total_amount"], 0, 100)

print("Morgan — Interaction (time_slot, hour_x_dayofweek) and key ratios (cbd_fee_ratio, surcharges_ratio, base_fare_ratio) added.")

Morgan — Interaction (time_slot, hour_x_dayofweek) and key ratios (cbd_fee_ratio, surcharges_ratio, base_fare_ratio) added.


## 6. Trim for size (target ≤100 MB) and modeling readiness

Drop redundant or heavy columns so the parquet stays ≤100 MB. We **keep** a few high-value columns for stakeholders: **pickup_borough** and **dropoff_borough** (who gets the CBD fee), **time_of_day** (when trips happen), and **distance_category** (short/medium/long) so the final feature set stays strong for trip duration and congestion-fee modeling.

In [32]:
# Drop columns that are redundant or heavy to get parquet ≤100 MB
# Kept for modeling: pickup_borough, dropoff_borough (CBD/congestion), time_of_day, distance_category
cols_to_drop = [
    "borough_route", "route_id", "pickup_day_name",
    # "pickup_borough", "dropoff_borough",  # KEEP: small cardinality; key for congestion (Manhattan/CBD)
    # "time_of_day",                       # KEEP: interpretable (morning/afternoon/evening/night)
    "passenger_count_binned",              # object; we have passenger_count
    # "distance_category",                 # KEEP: interpretable (short/medium/long)
    "pickup_zone_freq", "dropoff_zone_freq", "route_freq",  # 3 float64 (~60 MB); use location IDs in model
    "hour_sin", "hour_cos", "day_of_week_sin", "day_of_week_cos",  # cyclical; tree models use hour/dow
    "pickup_month", "pickup_day_of_month", # January only
    "is_peak_hour",                        # same as is_evening_rush
    "is_morning_rush", "is_evening_rush",  # keep is_rush_hour
    "is_jfk_pickup", "is_lga_pickup", "is_ewr_pickup",
    "is_airport_dropoff", "is_jfk_dropoff", "is_lga_dropoff", "is_ewr_dropoff",  # keep is_airport_pickup, is_airport_trip
    "fare_per_mile", "total_per_mile",     # derivable from fare_amount, total_amount, trip_distance
    "tip_percentage",                      # we have tip_to_total_ratio
    # "total_surcharges", "base_fare_ratio",  # KEEP: from Morgan_FE; useful for fare-structure modeling
    # "hour_x_dayofweek",                   # KEEP: from Morgan_FE; time interaction for models
    "store_and_fwd_flag",                  # we have store_and_fwd_encoded
    "tpep_dropoff_datetime",               # we have trip_duration_min and tpep_pickup_datetime
]
dropped = [c for c in cols_to_drop if c in df.columns]
df.drop(columns=dropped, inplace=True, errors="ignore")

# Downcast float64 to float32 to reduce parquet size (safe for this scale)
for col in df.select_dtypes(include=["float64"]).columns:
    df[col] = df[col].astype("float32")

print("Dropped for size:", len(dropped), "columns")
print(f"Columns after trim: {df.shape[1]}")

# Features added by each member (as created in Sections 2–5); count how many remain after trim
moses_added = ["pickup_hour", "pickup_day_of_week", "pickup_day_name", "pickup_month", "pickup_day_of_month",
               "is_weekend", "is_peak_hour", "is_morning_rush", "is_evening_rush", "is_rush_hour",
               "time_of_day", "hour_sin", "hour_cos", "day_of_week_sin", "day_of_week_cos"]
tarun_added = ["pickup_borough", "dropoff_borough", "is_same_borough", "borough_route",
               "is_airport_pickup", "is_jfk_pickup", "is_lga_pickup", "is_ewr_pickup",
               "is_airport_dropoff", "is_jfk_dropoff", "is_lga_dropoff", "is_ewr_dropoff",
               "is_airport_trip", "pickup_zone_freq", "dropoff_zone_freq", "route_id", "route_freq"]
abhishek_added = ["avg_speed_mph", "fare_per_mile", "total_per_mile", "is_single_passenger", "passenger_count_binned",
                  "tip_percentage", "tip_to_total_ratio", "distance_category", "is_extreme_distance", "is_extreme_fare"]
morgan_added = ["store_and_fwd_encoded", "payment_name", "ratecode_name", "time_slot", "hour_x_dayofweek",
                "cbd_fee_ratio", "total_surcharges", "surcharges_ratio", "base_fare_ratio"]

def retained(feat_list):
    return [c for c in feat_list if c in df.columns]

m_ret = retained(moses_added)
t_ret = retained(tarun_added)
a_ret = retained(abhishek_added)
mo_ret = retained(morgan_added)
total_ret = len(m_ret) + len(t_ret) + len(a_ret) + len(mo_ret)

print("\nFeatures by member (added in pipeline → retained in final dataset):")
print("  Moses:    ",  len(m_ret), "retained:", m_ret)
print("  Tarun:    ",  len(t_ret), "retained:", t_ret)
print("  Abhishek: ",  len(a_ret), "retained:", a_ret)
print("  Morgan:   ",  len(mo_ret), "retained:", mo_ret)
print("  Total engineered (retained):", total_ret)
print("\nAll retained engineered features:", sorted(m_ret + t_ret + a_ret + mo_ret))

Dropped for size: 0 columns
Columns after trim: 45

Features by member (added in pipeline → retained in final dataset):
  Moses:     5 retained: ['pickup_hour', 'pickup_day_of_week', 'is_weekend', 'is_rush_hour', 'time_of_day']
  Tarun:     5 retained: ['pickup_borough', 'dropoff_borough', 'is_same_borough', 'is_airport_pickup', 'is_airport_trip']
  Abhishek:  6 retained: ['avg_speed_mph', 'is_single_passenger', 'tip_to_total_ratio', 'distance_category', 'is_extreme_distance', 'is_extreme_fare']
  Morgan:    9 retained: ['store_and_fwd_encoded', 'payment_name', 'ratecode_name', 'time_slot', 'hour_x_dayofweek', 'cbd_fee_ratio', 'total_surcharges', 'surcharges_ratio', 'base_fare_ratio']
  Total engineered (retained): 25

All retained engineered features: ['avg_speed_mph', 'base_fare_ratio', 'cbd_fee_ratio', 'distance_category', 'dropoff_borough', 'hour_x_dayofweek', 'is_airport_pickup', 'is_airport_trip', 'is_extreme_distance', 'is_extreme_fare', 'is_rush_hour', 'is_same_borough', 'is_si

## 7. Export

Save the stakeholder-focused dataset to a single parquet file for modeling (trip duration and congestion fee prediction).

In [33]:
# Write single final parquet for modeling (compression for smaller file)
out_path = "../../data/processed/taxi_engineered.parquet"
df.to_parquet(out_path, index=False, compression="gzip")
print(f"Saved: {out_path}")
print(f"Shape: {df.shape[0]:,} x {df.shape[1]}")
import os
if os.path.exists(out_path):
    print(f"File size: {os.path.getsize(out_path) / (1024**2):.1f} MB")

Saved: ../../data/processed/taxi_engineered.parquet
Shape: 2,451,103 x 45
File size: 58.2 MB
